# Generate the cache for context-free cascades

Runs the small model (SLM), the large model (LLM) and every verifier once over a mixed
stream of GSM8K and TriviaQA questions, then grades everything into `cache.jsonl`.
The router experiments run offline from that file.

**Kaggle:** Settings → Accelerator **GPU T4 x2** (vLLM does not support the P100) and
**Internet on** (needs a phone-verified account). Then *Save Version → Save & Run All* so it
runs unattended and the outputs persist.
**Colab:** Runtime → Change runtime type → **T4 GPU**.

Set `PILOT = True` first (200 queries, about 10 minutes) to check everything works, then
`PILOT = False` for the full run. Every stage resumes if the session drops.
Download `cache.zip` (Kaggle: Output tab; Colab: the file browser) when it finishes.

In [ ]:
PILOT = True                                   # 100 math + 100 factual; False = 1,319 + 1,500
SLM = "Qwen/Qwen2.5-1.5B-Instruct"
LLM = "Qwen/Qwen2.5-7B-Instruct-AWQ"           # 4-bit, fits one T4 alongside nothing else
BACKEND = "vllm"                               # "hf" = plain transformers fallback (much slower)
DTYPE = "half"                                 # T4 has no bfloat16
STAGES = ["prepare", "slm", "retrieve", "verify", "llm", "embed", "merge"]
WIKI_CONTACT = ""                              # your email or URL: Wikipedia allows 200 req/min with it, 10 without

import os, subprocess
cap = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip().splitlines()
print("GPU:", cap)
NEEDS_VLLM = BACKEND == "vllm" and any(s in STAGES for s in ("slm", "verify", "llm"))
if NEEDS_VLLM and (not cap or float(cap[0].split(",")[-1]) < 7.0):
    raise SystemExit("vLLM needs a T4 or newer GPU (compute capability >= 7.0). "
                     "Kaggle: Settings -> Accelerator -> GPU T4 x2. Colab: Runtime -> T4 GPU.")
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
OUT = f"{WORK}/cache_pilot" if PILOT else f"{WORK}/cache"
LIMITS = "--limit-math 100 --limit-factual 100" if PILOT else "--limit-factual 1500"
os.chdir(WORK)
os.environ["PYTHONPATH"] = f"{WORK}/src"
os.environ["WIKI_CONTACT"] = WIKI_CONTACT

# Outputs of earlier stages attached as a Kaggle dataset are copied in; those stages then skip.
import glob, shutil
os.makedirs(OUT, exist_ok=True)
STAGE_FILES = ["queries.jsonl", "slm.jsonl", "retrieval.jsonl", "verify.jsonl", "llm.jsonl",
               "embeddings.npy", "embeddings_qids.json", "cache.jsonl", "hidden.npz"]
for path in glob.glob("/kaggle/input/**/*", recursive=True):
    name = os.path.basename(path)
    if name in STAGE_FILES and not os.path.exists(f"{OUT}/{name}"):
        shutil.copy(path, f"{OUT}/{name}")
        print("using input", path)
print(OUT, STAGES)

## Install

In [ ]:
import subprocess, sys
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)
pip = f"{sys.executable} -m pip"
if NEEDS_VLLM:
    subprocess.run(f"{pip} install -q vllm sentence-transformers datasets requests", shell=True, check=True)
    # vLLM brings its own torch build; the image's torchaudio no longer matches it and breaks
    # `import transformers`. Nothing here needs torchaudio.
    subprocess.run(f"{pip} uninstall -y -q torchaudio", shell=True)
else:  # hidden-state stage only: the image's torch + transformers suffice
    subprocess.run(f"{pip} install -q sentence-transformers datasets requests", shell=True, check=True)

In [ ]:
import subprocess, sys
probe = ("import vllm; print('vllm', vllm.__version__); " if NEEDS_VLLM else "") +     "import torch, transformers, sentence_transformers; print('torch', torch.__version__, 'transformers', transformers.__version__, 'cuda', torch.cuda.is_available())"
r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
print(r.stdout, r.stderr[-3000:])
assert r.returncode == 0, "environment check failed, see above"

os.environ["TQDM_MININTERVAL"] = "30"   # keep progress bars from flooding the log

def run(args):
    """Run one pipeline stage, stream its output, and stop the notebook if it fails."""
    cmd = f"{sys.executable} -m vcascade.generate --out {OUT} {args}"
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"stage failed: {args}")

## Write the `vcascade` package
Generated from `src/vcascade/`; do not edit here.

In [ ]:
os.makedirs("src/vcascade", exist_ok=True)  # relative to WORK, set above

In [ ]:
%%writefile src/vcascade/__init__.py
"""Cost-aware verifier selection for context-free LLM cascades.

Generation side (runs on a GPU box: Kaggle / Colab):
    data, prompts, backends, retrieval, generate
Router side (runs anywhere, CPU only, from the cached outputs):
    grading, signals, costs, belief, routers, evaluate, simulate
"""

TYPES = ("easy", "hard", "unsolvable", "slm_only")
"""Latent query types, keyed by (SLM correct, LLM correct):
easy = (1, 1), hard = (0, 1), unsolvable = (0, 0), slm_only = (1, 0)."""

In [ ]:
%%writefile src/vcascade/data.py
"""Load benchmark datasets into one shuffled query stream.

Every query is a plain dict so it serialises straight to JSONL:
    {"qid": "gsm8k-0012", "task": "math", "question": "...", "gold": {...}}
The `task` field is only used for grading, analysis and the oracle
per-category baseline. Routers never see it.
"""

from __future__ import annotations

import random


def load_gsm8k(limit: int | None = None) -> list[dict]:
    from datasets import load_dataset

    rows = load_dataset("openai/gsm8k", "main", split="test")
    out = []
    for i, r in enumerate(rows):
        final = r["answer"].split("####")[-1].strip().replace(",", "")
        out.append({
            "qid": f"gsm8k-{i:04d}",
            "task": "math",
            "question": r["question"].strip(),
            "gold": {"answer": final},
        })
    return out[:limit] if limit else out


def load_triviaqa(limit: int = 1500, seed: int = 0) -> list[dict]:
    from datasets import load_dataset

    rows = load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="validation")
    first = {}
    for i, qid in enumerate(rows["question_id"]):  # the split repeats some questions
        first.setdefault(qid, i)
    idx = sorted(first.values())
    random.Random(seed).shuffle(idx)
    out = []
    for i in idx[:limit]:
        r = rows[i]
        out.append({
            "qid": f"tqa-{r['question_id']}",
            "task": "factual",
            "question": r["question"].strip(),
            "gold": {
                "answer": r["answer"]["value"],
                "aliases": sorted(set(r["answer"]["normalized_aliases"])),
            },
        })
    return out


LOADERS = {"math": load_gsm8k, "factual": load_triviaqa}


def build_stream(tasks=("math", "factual"), limits: dict | None = None, seed: int = 0) -> list[dict]:
    """Concatenate the requested tasks and shuffle them into one stream."""
    limits = limits or {}
    queries = []
    for t in tasks:
        queries += LOADERS[t](limit=limits.get(t)) if limits.get(t) else LOADERS[t]()
    random.Random(seed).shuffle(queries)
    return queries

In [ ]:
%%writefile src/vcascade/prompts.py
"""Prompts for answering and for the two LLM-based verifiers (V3, V4).

One generic answer prompt serves every query: a deployed assistant does not
know the task type in advance, so nothing here depends on it.

The verification prompts follow AutoMix's entailment-style prompt
(Aggarwal et al., 2024, Listing 1). V3 adds retrieved Wikipedia passages as
the context; V4 is the same prompt with no context at all.
"""

from __future__ import annotations

ANSWER_PROMPT = (
    "Answer the following question. If it needs calculation or reasoning, think step by step first; "
    "if it is a factual question, answer directly in a few words. "
    "End your response with a final line of the form 'Final answer: <answer>', where <answer> is "
    "only the answer itself (a number, a name or a short phrase), not a sentence.\n\n"
    "Question: {question}"
)
MAX_ANSWER_TOKENS = 512

_VERIFY_INSTRUCTION = (
    "Instruction: Your task is to evaluate if the AI Generated Answer is correct{basis}. "
    "Think briefly, then finish with exactly one line: 'Verdict: Correct' or 'Verdict: Incorrect'."
)

VERIFY_WITH_CONTEXT = (
    "Context: {context}\n\n"
    "Question: {question}\n\n"
    "AI Generated Answer: {answer}\n\n"
    + _VERIFY_INSTRUCTION.format(basis=", based on the provided context and question")
    + "\n\nEvaluation:"
)

VERIFY_NO_CONTEXT = (
    "Question: {question}\n\n"
    "AI Generated Answer: {answer}\n\n"
    + _VERIFY_INSTRUCTION.format(basis="")
    + "\n\nEvaluation:"
)
MAX_VERIFY_TOKENS = 256


def answer_messages(query: dict) -> list[dict]:
    return [{"role": "user", "content": ANSWER_PROMPT.format(question=query["question"])}]


def verify_messages(query: dict, answer_text: str, context: str | None) -> list[dict]:
    answer_text = answer_text.strip()[-1500:]
    if context is None:
        content = VERIFY_NO_CONTEXT.format(question=query["question"], answer=answer_text)
    else:
        content = VERIFY_WITH_CONTEXT.format(context=context, question=query["question"], answer=answer_text)
    return [{"role": "user", "content": content}]

In [ ]:
%%writefile src/vcascade/grading.py
"""Answer extraction, answer comparison and correctness checks.

Extraction and comparison are task-agnostic (they feed router signals such as
self-consistency). Only `is_correct` reads the gold answer and the task.
"""

from __future__ import annotations

import re
import string

_NUM = re.compile(r"-?\d[\d,]*\.?\d*")
_FINAL = re.compile(r"final answer\s*[:：]\s*(.+)", flags=re.IGNORECASE)


def extract_answer(text: str) -> str | None:
    """Text after the last 'Final answer:'; falls back to the last non-empty line."""
    hits = _FINAL.findall(text)
    if hits:
        ans = hits[-1]
    else:
        lines = [ln for ln in text.strip().splitlines() if ln.strip()]
        if not lines:
            return None
        ans = lines[-1]
    ans = ans.strip().strip("*").strip()
    return ans or None


def canon_number(s: str) -> str | None:
    s = s.replace(",", "").replace("$", "").strip().rstrip(".")
    try:
        v = float(s)
    except ValueError:
        return None
    if v != v or v in (float("inf"), float("-inf")):  # absurdly long digit strings overflow
        return None
    return str(int(v)) if v == int(v) else f"{v:g}"


def normalize_text(s: str) -> str:
    """TriviaQA / SQuAD normalisation: lowercase, drop punctuation and articles."""
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def canonical(answer: str | None) -> str | None:
    """Comparison key for two answers: a number if the answer is numeric, else normalised text."""
    if answer is None:
        return None
    nums = _NUM.findall(answer)
    # "18", "$18.00" and "18 eggs" all compare as 18; longer text compares as text.
    if len(nums) == 1 and len(answer.split()) <= 4:
        num = canon_number(nums[0])
        if num is not None:
            return num
    return normalize_text(answer)


MAX_ANSWER_WORDS = 15  # longer predictions are not credited by containment (they may hedge)


def _contains_phrase(text: str, phrase: str) -> bool:
    """Whole-word containment of one normalised phrase in another."""
    return bool(phrase) and f" {phrase} " in f" {text} "


def answers_match(a: str | None, b: str | None) -> bool:
    """Do two extracted answers say the same thing? Used for self-consistency (V2).

    Numbers compare numerically; text compares after normalisation, also accepting
    one answer contained in the other ("Benedict Cumberbatch" vs "played by Benedict Cumberbatch").
    """
    ka, kb = canonical(a), canonical(b)
    if ka is None or kb is None:
        return False
    if ka == kb:
        return True
    if canon_number(ka) is not None or canon_number(kb) is not None:
        return False
    short, long_ = sorted((ka, kb), key=len)
    return len(long_.split()) <= MAX_ANSWER_WORDS and _contains_phrase(long_, short)


def is_correct(query: dict, text: str) -> bool:
    pred = extract_answer(text)
    if pred is None:
        return False
    gold = query["gold"]
    if query["task"] == "math":
        nums = _NUM.findall(pred)
        return bool(nums) and canon_number(nums[-1]) == canon_number(gold["answer"])
    # TriviaQA: exact match on normalised aliases, or an alias as whole words inside a short answer
    p = normalize_text(pred)
    if p in set(gold["aliases"]):
        return True
    return len(p.split()) <= MAX_ANSWER_WORDS and any(_contains_phrase(p, a) for a in gold["aliases"])


def parse_verdict(text: str) -> int | None:
    """1 = Correct, 0 = Incorrect, None = no verdict found. The last mention wins."""
    hits = re.findall(r"\b(incorrect|correct)\b", text, flags=re.IGNORECASE)
    if not hits:
        return None
    return 0 if hits[-1].lower() == "incorrect" else 1

In [ ]:
%%writefile src/vcascade/backends.py
"""Model backends with one interface.

    backend.generate(conversations, n=..., temperature=..., max_tokens=..., logprobs=...)
        -> one list per conversation, holding n completions of the form
           {"text": str, "token_logprobs": list[float] | None,
            "prompt_tokens": int, "completion_tokens": int}

VLLMBackend runs a local model on the GPU (Kaggle / Colab).
OpenAICompatBackend calls any OpenAI-compatible endpoint (Together, Groq,
OpenRouter, or a vLLM server).
"""

from __future__ import annotations

import os
import time
from concurrent.futures import ThreadPoolExecutor


class VLLMBackend:
    def __init__(self, model: str, dtype: str = "auto", tensor_parallel_size: int = 1,
                 max_model_len: int = 4096, gpu_memory_utilization: float = 0.90, seed: int = 0):
        from vllm import LLM

        self.model = model
        self.llm = LLM(model=model, dtype=dtype, tensor_parallel_size=tensor_parallel_size,
                       max_model_len=max_model_len, gpu_memory_utilization=gpu_memory_utilization,
                       seed=seed, enforce_eager=False)

    def generate(self, conversations, n=1, temperature=0.0, max_tokens=512, logprobs=False, seed=0):
        from vllm import SamplingParams

        sp = SamplingParams(n=n, temperature=temperature, top_p=1.0 if temperature == 0 else 0.95,
                            max_tokens=max_tokens, logprobs=1 if logprobs else None, seed=seed)
        outs = self.llm.chat(conversations, sp, use_tqdm=True)
        results = []
        for req in outs:
            comps = []
            for c in req.outputs:
                lps = None
                if logprobs and c.logprobs is not None:
                    lps = [step[tok].logprob for tok, step in zip(c.token_ids, c.logprobs)]
                comps.append({"text": c.text, "token_logprobs": lps,
                              "prompt_tokens": len(req.prompt_token_ids), "completion_tokens": len(c.token_ids)})
            results.append(comps)
        return results


class OpenAICompatBackend:
    def __init__(self, model: str, base_url: str, api_key_env: str = "LLM_API_KEY",
                 max_workers: int = 8, max_retries: int = 6):
        from openai import OpenAI

        self.model = model
        self.client = OpenAI(base_url=base_url, api_key=os.environ[api_key_env])
        self.max_workers = max_workers
        self.max_retries = max_retries

    def _one(self, messages, temperature, max_tokens, logprobs, seed):
        for attempt in range(self.max_retries):
            try:
                r = self.client.chat.completions.create(
                    model=self.model, messages=messages, temperature=temperature,
                    max_tokens=max_tokens, logprobs=logprobs or None, seed=seed)
                ch = r.choices[0]
                lps = [t.logprob for t in ch.logprobs.content] if logprobs and ch.logprobs else None
                return {"text": ch.message.content or "", "token_logprobs": lps,
                        "prompt_tokens": r.usage.prompt_tokens, "completion_tokens": r.usage.completion_tokens}
            except Exception:  # rate limits and transient errors: back off and retry
                if attempt == self.max_retries - 1:
                    raise
                time.sleep(2 ** attempt)

    def generate(self, conversations, n=1, temperature=0.0, max_tokens=512, logprobs=False, seed=0):
        # Many providers reject n > 1, so samples are separate requests.
        jobs = [(m, s) for m in conversations for s in range(n)]
        with ThreadPoolExecutor(self.max_workers) as pool:
            flat = list(pool.map(lambda j: self._one(j[0], temperature, max_tokens, logprobs, seed + j[1]), jobs))
        return [flat[i * n:(i + 1) * n] for i in range(len(conversations))]


class HFBackend:
    """Plain transformers fallback, for when vLLM will not install. Slower, works on any GPU."""

    def __init__(self, model: str, dtype: str = "auto", batch_size: int = 16, **_):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.model = model
        self.tok = AutoTokenizer.from_pretrained(model, padding_side="left")
        torch_dtype = {"auto": "auto", "half": torch.float16, "float16": torch.float16,
                       "bfloat16": torch.bfloat16}[dtype]
        self.lm = AutoModelForCausalLM.from_pretrained(model, torch_dtype=torch_dtype, device_map="auto").eval()
        self.batch_size = batch_size

    def generate(self, conversations, n=1, temperature=0.0, max_tokens=512, logprobs=False, seed=0):
        import torch
        from tqdm.auto import tqdm

        torch.manual_seed(seed)
        eos = {self.tok.eos_token_id, self.tok.pad_token_id}
        results = []
        bs = max(1, self.batch_size // n)
        for start in tqdm(range(0, len(conversations), bs)):
            texts = [self.tok.apply_chat_template(c, add_generation_prompt=True, tokenize=False)
                     for c in conversations[start:start + bs]]
            enc = self.tok(texts, return_tensors="pt", padding=True).to(self.lm.device)
            sample = temperature > 0
            with torch.no_grad():
                out = self.lm.generate(**enc, max_new_tokens=max_tokens, do_sample=sample,
                                       temperature=temperature if sample else None, top_p=0.95 if sample else None,
                                       num_return_sequences=n, return_dict_in_generate=True,
                                       output_scores=logprobs, pad_token_id=self.tok.pad_token_id)
            gen = out.sequences[:, enc["input_ids"].shape[1]:]
            lps = self.lm.compute_transition_scores(out.sequences, out.scores, normalize_logits=True) \
                if logprobs else None
            for j in range(len(texts)):
                comps = []
                for r in range(n):
                    row = j * n + r
                    ids = gen[row].tolist()
                    length = next((t for t, x in enumerate(ids) if x in eos), len(ids))
                    comps.append({
                        "text": self.tok.decode(ids[:length], skip_special_tokens=True),
                        "token_logprobs": lps[row, :length].float().tolist() if logprobs else None,
                        "prompt_tokens": int(enc["attention_mask"][j].sum()), "completion_tokens": length})
                results.append(comps)
        return results


def make_backend(spec: dict):
    kind = spec.pop("kind")
    return {"vllm": VLLMBackend, "openai": OpenAICompatBackend, "hf": HFBackend}[kind](**spec)

In [ ]:
%%writefile src/vcascade/retrieval.py
"""Evidence retrieval for the retrieval-grounded verifier (V3).

Uses the public Wikipedia API: no index to download, works from a Kaggle /
Colab notebook with internet enabled. The search string is the question plus
the SLM's candidate answer, so the passages bear on the claim being checked.

Wikimedia allows 200 requests/minute to a client whose User-Agent carries contact
info (set WIKI_CONTACT) and at most 3 concurrent requests. We use 3 workers behind
a shared limiter of 3 requests/second, retry 429s with backoff (honouring
Retry-After), and append progress to a resume file so an interrupted run picks up
where it stopped.
"""

from __future__ import annotations

import json
import os
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import requests

API = "https://en.wikipedia.org/w/api.php"
# Wikimedia's User-Agent policy wants contact info (an email or a URL); without it requests
# fall in the "unidentified" tier, limited to 10 per minute instead of 200.
CONTACT = os.environ.get("WIKI_CONTACT", "")
HEADERS = {"User-Agent": f"vcascade-research-bot/0.1 ({CONTACT}) python-requests" if CONTACT
           else "vcascade-research-bot/0.1 python-requests"}


_rate_lock = threading.Lock()
_last_request = [0.0]
MIN_INTERVAL = 1 / 3  # seconds between requests across all workers: 180/min, under the 200/min limit


def _throttle():
    with _rate_lock:
        wait = _last_request[0] + MIN_INTERVAL - time.monotonic()
        if wait > 0:
            time.sleep(wait)
        _last_request[0] = time.monotonic()


def _get(params: dict, retries: int = 8) -> dict:
    for attempt in range(retries):
        _throttle()
        try:
            r = requests.get(API, params={**params, "format": "json"}, headers=HEADERS, timeout=20)
            if r.status_code == 429 or r.status_code >= 500:
                wait = float(r.headers.get("Retry-After", 0) or 0) or min(2 ** attempt * 2, 120)
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r.json()
        except requests.RequestException:
            time.sleep(min(2 ** attempt * 2, 120))
    raise RuntimeError(f"Wikipedia API kept failing after {retries} attempts")


def clean_query(text: str, max_words: int = 30) -> str:
    """Drop quotes and search-operator punctuation; long or quoted queries otherwise match nothing."""
    text = re.sub(r"[\"“”‘’'`?!:;()\[\]{}<>|~*^\/=+#]", " ", text)
    return " ".join(text.split()[:max_words])


_STOP = set("""a an the of in on at to for from by with and or but is are was were be been being which what who
whom whose when where why how that this these those it its as into than then there their they he she his her
him we you your i do does did has have had not no all any each some such can could would should will may might
also especially traditionally often usually known called name named""".split())


def keywords(text: str, max_words: int = 10) -> str:
    words = re.sub(r"[,.]", " ", clean_query(text, 60)).split()
    return " ".join([w for w in words if w.lower() not in _STOP][:max_words])


def search_passages(query: str, k: int = 3, chars: int = 700, fallback: str | None = None) -> list[dict]:
    """Top-k search hits with their intro extracts, one request per attempt.

    Tries the query, then `fallback` (the question alone), then the fallback's keywords:
    Wikipedia search requires most terms to match, so long questions can find nothing.
    """
    attempts = [clean_query(query)] + ([clean_query(fallback), keywords(fallback)] if fallback else [])
    for q in dict.fromkeys(a for a in attempts if a):  # dedupe, keep order
        out = _search(q, k, chars)
        if out:
            return out
    return []


def _search(query: str, k: int, chars: int) -> list[dict]:
    if not query:
        return []
    data = _get({"action": "query", "generator": "search", "gsrsearch": query[:300], "gsrlimit": k,
                 "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": k})
    pages = sorted(data.get("query", {}).get("pages", {}).values(), key=lambda p: p.get("index", 0))
    out = []
    for p in pages:
        text = " ".join((p.get("extract") or "").split())
        if text:
            out.append({"title": p.get("title", ""), "text": text[:chars]})
    return out


def format_context(passages: list[dict]) -> str:
    if not passages:
        return "(no relevant passages found)"
    return "\n\n".join(f"[{i + 1}] {p['title']}: {p['text']}" for i, p in enumerate(passages))


def retrieve_all(queries: list[str], k: int = 3, resume_file: str | Path | None = None,
                 max_fail_frac: float = 0.02, fallbacks: list[str] | None = None,
                 workers: int = 3) -> list[list[dict]]:
    done: dict[str, list] = {}
    if resume_file and Path(resume_file).exists():
        with open(resume_file, encoding="utf-8") as f:
            for line in f:
                r = json.loads(line)
                done[r["search"]] = r["passages"]
    todo = [i for i, q in enumerate(queries) if q not in done]
    lock = threading.Lock()
    state = {"fails": 0, "n": 0}
    log = open(resume_file, "a", encoding="utf-8") if resume_file else None

    def work(i):
        q = queries[i]
        try:
            res = search_passages(q, k=k, fallback=fallbacks[i] if fallbacks else None)
        except RuntimeError:
            with lock:
                state["fails"] += 1
                if state["fails"] > max_fail_frac * len(queries):
                    raise
            res = []  # V3 then sees "(no relevant passages found)"
        with lock:
            done[q] = res
            if log:
                log.write(json.dumps({"search": q, "passages": res}, ensure_ascii=False) + "\n")
                log.flush()
            state["n"] += 1
            if state["n"] % 200 == 0:
                print(f"retrieved {state['n']}/{len(todo)} ({state['fails']} failed)", flush=True)

    try:
        with ThreadPoolExecutor(workers) as pool:
            list(pool.map(work, todo))
    finally:
        if log:
            log.close()
    return [done[q] for q in queries]

In [ ]:
%%writefile src/vcascade/generate.py
"""Generation pipeline: produce every model output and verifier signal once.

Each stage writes one file in --out and is skipped if that file exists, so a
dropped Kaggle / Colab session resumes where it stopped. Run the GPU stages as
separate processes so each model's GPU memory is released on exit.

    python -m vcascade.generate --out cache prepare --limit-math 1319 --limit-factual 1500
    python -m vcascade.generate --out cache slm      --model Qwen/Qwen2.5-1.5B-Instruct
    python -m vcascade.generate --out cache retrieve
    python -m vcascade.generate --out cache verify   --model Qwen/Qwen2.5-1.5B-Instruct
    python -m vcascade.generate --out cache llm      --model Qwen/Qwen2.5-7B-Instruct-AWQ
    python -m vcascade.generate --out cache embed
    python -m vcascade.generate --out cache merge
    python -m vcascade.generate --out cache hidden   --model Qwen/Qwen2.5-1.5B-Instruct   # probe signal
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

from . import grading, prompts


def read_jsonl(path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(path, rows) -> None:
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    tmp.replace(path)  # atomic: a killed session never leaves a half-written stage file


def _backend(args):
    from .backends import make_backend

    if args.backend == "vllm":
        return make_backend({"kind": "vllm", "model": args.model, "dtype": args.dtype,
                             "tensor_parallel_size": args.tp, "max_model_len": args.max_model_len})
    if args.backend == "hf":
        return make_backend({"kind": "hf", "model": args.model, "dtype": args.dtype})
    return make_backend({"kind": "openai", "model": args.model, "base_url": args.base_url})


def stage_prepare(args, out: Path):
    from .data import build_stream

    queries = build_stream(limits={"math": args.limit_math, "factual": args.limit_factual}, seed=args.seed)
    write_jsonl(out / "queries.jsonl", queries)
    print(f"{len(queries)} queries written")


def stage_slm(args, out: Path):
    queries = read_jsonl(out / "queries.jsonl")
    be = _backend(args)
    convs = [prompts.answer_messages(q) for q in queries]
    greedy = be.generate(convs, n=1, temperature=0.0, max_tokens=prompts.MAX_ANSWER_TOKENS, logprobs=True)
    samples = be.generate(convs, n=args.n_samples, temperature=0.7,
                          max_tokens=prompts.MAX_ANSWER_TOKENS, seed=args.seed)
    write_jsonl(out / "slm.jsonl", [
        {"qid": q["qid"], "model": args.model, "greedy": g[0], "samples": s}
        for q, g, s in zip(queries, greedy, samples)])


def stage_retrieve(args, out: Path):
    from .retrieval import retrieve_all

    queries = read_jsonl(out / "queries.jsonl")
    slm = {r["qid"]: r for r in read_jsonl(out / "slm.jsonl")}
    search = []
    for q in queries:
        ans = grading.extract_answer(slm[q["qid"]]["greedy"]["text"]) or ""
        search.append(f"{q['question']} {ans}")
    passages = retrieve_all(search, k=args.k_passages, resume_file=out / "retrieval.partial.jsonl",
                            fallbacks=[q["question"] for q in queries])
    write_jsonl(out / "retrieval.jsonl", [
        {"qid": q["qid"], "search": s, "passages": p} for q, s, p in zip(queries, search, passages)])


def stage_verify(args, out: Path):
    from .retrieval import format_context

    queries = read_jsonl(out / "queries.jsonl")
    slm = {r["qid"]: r for r in read_jsonl(out / "slm.jsonl")}
    ret = {r["qid"]: r for r in read_jsonl(out / "retrieval.jsonl")}
    be = _backend(args)
    kw = dict(n=args.n_verify, temperature=0.7, max_tokens=prompts.MAX_VERIFY_TOKENS, seed=args.seed)
    v3 = be.generate([prompts.verify_messages(q, slm[q["qid"]]["greedy"]["text"],
                                              format_context(ret[q["qid"]]["passages"])) for q in queries], **kw)
    v4 = be.generate([prompts.verify_messages(q, slm[q["qid"]]["greedy"]["text"], None) for q in queries], **kw)
    write_jsonl(out / "verify.jsonl", [
        {"qid": q["qid"], "v3": a, "v4": b} for q, a, b in zip(queries, v3, v4)])


def stage_llm(args, out: Path):
    queries = read_jsonl(out / "queries.jsonl")
    be = _backend(args)
    greedy = be.generate([prompts.answer_messages(q) for q in queries], n=1, temperature=0.0,
                         max_tokens=prompts.MAX_ANSWER_TOKENS)
    write_jsonl(out / "llm.jsonl", [
        {"qid": q["qid"], "model": args.model, "greedy": g[0]} for q, g in zip(queries, greedy)])


def stage_embed(args, out: Path):
    import numpy as np
    from sentence_transformers import SentenceTransformer

    queries = read_jsonl(out / "queries.jsonl")
    model = SentenceTransformer(args.embed_model)
    emb = model.encode([q["question"] for q in queries], batch_size=128, normalize_embeddings=True,
                       show_progress_bar=True)
    np.save(out / "embeddings.npy", emb.astype("float32"))
    (out / "embeddings_qids.json").write_text(json.dumps([q["qid"] for q in queries]))


def stage_hidden(args, out: Path):
    """SLM hidden states for the probe signal: one forward pass over prompt + greedy answer.

    Saves, at four depths (1/4, 1/2, 3/4, last layer), float16 arrays of shape (n, 4, d):
        prompt  state at the last prompt token, before any answer exists (query-only)
        last    state at the last answer token
        mean    mean state over the answer tokens
    """
    import numpy as np
    import torch
    from tqdm.auto import tqdm
    from transformers import AutoModelForCausalLM, AutoTokenizer

    queries, seen = [], set()
    for q in read_jsonl(out / "queries.jsonl"):
        if q["qid"] not in seen:
            seen.add(q["qid"])
            queries.append(q)
    slm = {r["qid"]: r for r in read_jsonl(out / "slm.jsonl")}
    tok = AutoTokenizer.from_pretrained(args.model)
    dtype = torch.float16 if args.dtype in ("half", "float16", "auto") else torch.bfloat16
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    if dev == "cpu":
        dtype = torch.float32
    lm = AutoModelForCausalLM.from_pretrained(args.model, torch_dtype=dtype).to(dev).eval()
    n_layers = lm.config.num_hidden_layers
    layers = [n_layers // 4, n_layers // 2, 3 * n_layers // 4, n_layers]  # hidden_states[0] = embeddings

    seqs = []  # (prompt length, token ids), truncated to max_model_len
    for q in queries:
        prompt = tok.apply_chat_template(prompts.answer_messages(q), add_generation_prompt=True, tokenize=False)
        p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
        a_ids = tok(slm[q["qid"]]["greedy"]["text"], add_special_tokens=False)["input_ids"] or [tok.eos_token_id]
        seqs.append((len(p_ids), (p_ids + a_ids)[:args.max_model_len]))

    d = lm.config.hidden_size
    feats = {k: np.zeros((len(seqs), len(layers), d), dtype=np.float16) for k in ("prompt", "last", "mean")}
    order = np.argsort([len(s) for _, s in seqs])  # similar lengths per batch: less padding
    bs = args.batch_size
    for start in tqdm(range(0, len(order), bs)):
        idx = order[start:start + bs]
        width = max(len(seqs[i][1]) for i in idx)
        ids = torch.full((len(idx), width), tok.pad_token_id or 0, dtype=torch.long)
        mask = torch.zeros((len(idx), width), dtype=torch.long)
        for j, i in enumerate(idx):  # right padding keeps positions simple
            s = seqs[i][1]
            ids[j, :len(s)], mask[j, :len(s)] = torch.tensor(s), 1
        with torch.no_grad():
            hs = lm(input_ids=ids.to(dev), attention_mask=mask.to(dev), output_hidden_states=True).hidden_states
        for li, layer in enumerate(layers):
            h = hs[layer].float()
            for j, i in enumerate(idx):
                p, length = seqs[i][0], len(seqs[i][1])
                p = min(p, length - 1)  # answer truncated away entirely: fall back to the prompt
                feats["prompt"][i, li] = h[j, p - 1].cpu().numpy()
                feats["last"][i, li] = h[j, length - 1].cpu().numpy()
                feats["mean"][i, li] = h[j, p:length].mean(0).cpu().numpy()
    np.savez_compressed(out / "hidden.npz", qids=np.array([q["qid"] for q in queries]), layers=np.array(layers),
                        **feats)
    print(f"hidden states for {len(queries)} queries, layers {layers}, dim {d}")


def stage_merge(args, out: Path):
    """Grade everything and write cache.jsonl: the only file the router side needs."""
    import math

    queries = read_jsonl(out / "queries.jsonl")
    slm = {r["qid"]: r for r in read_jsonl(out / "slm.jsonl")}
    llm = {r["qid"]: r for r in read_jsonl(out / "llm.jsonl")}
    ver = {r["qid"]: r for r in read_jsonl(out / "verify.jsonl")}
    rows, seen = [], set()
    for q in queries:
        if q["qid"] in seen:  # TriviaQA repeats some question ids; keep one copy
            continue
        seen.add(q["qid"])
        s, l, v = slm[q["qid"]], llm[q["qid"]], ver[q["qid"]]
        g = s["greedy"]
        lps = g["token_logprobs"] or [0.0]
        greedy_ans = grading.extract_answer(g["text"])
        rows.append({
            "qid": q["qid"],
            "task": q["task"],
            "question": q["question"],
            "slm_answer": grading.extract_answer(g["text"]),
            "llm_answer": grading.extract_answer(l["greedy"]["text"]),
            "slm_correct": int(grading.is_correct(q, g["text"])),
            "llm_correct": int(grading.is_correct(q, l["greedy"]["text"])),
            # V1: confidence of the greedy answer, free with generation
            "v1": {"mean_lp": sum(lps) / len(lps), "min_lp": min(lps),
                   "perplexity": math.exp(-sum(lps) / len(lps))},
            # V2: does each temperature sample agree with the greedy answer?
            "v2": [int(grading.answers_match(grading.extract_answer(c["text"]), greedy_ans))
                   for c in s["samples"]],
            # V3 / V4: verdict votes, 1 = Correct, 0 = Incorrect, None = unparseable
            "v3": [grading.parse_verdict(c["text"]) for c in v["v3"]],
            "v4": [grading.parse_verdict(c["text"]) for c in v["v4"]],
            "tokens": {
                "slm_in": g["prompt_tokens"], "slm_out": g["completion_tokens"],
                "sample_out": sum(c["completion_tokens"] for c in s["samples"]) / max(len(s["samples"]), 1),
                "v3_in": v["v3"][0]["prompt_tokens"], "v3_out": sum(c["completion_tokens"] for c in v["v3"]),
                "v4_in": v["v4"][0]["prompt_tokens"], "v4_out": sum(c["completion_tokens"] for c in v["v4"]),
                "llm_in": l["greedy"]["prompt_tokens"], "llm_out": l["greedy"]["completion_tokens"],
            },
        })
    write_jsonl(out / "cache.jsonl", rows)
    n = len(rows)
    for task in sorted({r["task"] for r in rows}):
        sub = [r for r in rows if r["task"] == task]
        print(f"{task:8s} n={len(sub):5d}  SLM acc={sum(r['slm_correct'] for r in sub) / len(sub):.3f}"
              f"  LLM acc={sum(r['llm_correct'] for r in sub) / len(sub):.3f}")
    print(f"all      n={n:5d}  SLM acc={sum(r['slm_correct'] for r in rows) / n:.3f}"
          f"  LLM acc={sum(r['llm_correct'] for r in rows) / n:.3f}")


STAGES = {"prepare": (stage_prepare, "queries.jsonl"), "slm": (stage_slm, "slm.jsonl"),
          "retrieve": (stage_retrieve, "retrieval.jsonl"), "verify": (stage_verify, "verify.jsonl"),
          "llm": (stage_llm, "llm.jsonl"), "embed": (stage_embed, "embeddings.npy"),
          "merge": (stage_merge, "cache.jsonl"), "hidden": (stage_hidden, "hidden.npz")}


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("stage", choices=STAGES)
    p.add_argument("--out", default="cache")
    p.add_argument("--force", action="store_true", help="rerun even if the stage's output exists")
    p.add_argument("--model", default="Qwen/Qwen2.5-1.5B-Instruct")
    p.add_argument("--backend", choices=["vllm", "hf", "openai"], default="vllm")
    p.add_argument("--base-url", default=None, help="OpenAI-compatible endpoint, for --backend openai")
    p.add_argument("--dtype", default="auto", help="'half' on T4 / P100 GPUs, which lack bfloat16")
    p.add_argument("--tp", type=int, default=1, help="tensor parallel size (number of GPUs)")
    p.add_argument("--max-model-len", type=int, default=4096)
    p.add_argument("--limit-math", type=int, default=None)
    p.add_argument("--limit-factual", type=int, default=1500)
    p.add_argument("--n-samples", type=int, default=8)
    p.add_argument("--n-verify", type=int, default=4)
    p.add_argument("--k-passages", type=int, default=3)
    p.add_argument("--embed-model", default="BAAI/bge-small-en-v1.5")
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--batch-size", type=int, default=8, help="forward-pass batch for the hidden stage")
    args = p.parse_args(argv)

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    fn, product = STAGES[args.stage]
    if (out / product).exists() and not args.force:
        print(f"{args.stage}: {out / product} exists, skipping (use --force to rerun)")
        return
    fn(args, out)


if __name__ == "__main__":
    main()

## Run the pipeline

### Build the query stream

In [ ]:
if "prepare" in STAGES:
    run(f"prepare {LIMITS}")

### SLM: greedy answer with token log-probs (V1) + 8 samples (V2)

In [ ]:
if "slm" in STAGES:
    run(f"slm --model {SLM} --backend {BACKEND} --dtype {DTYPE}")

### Retrieve Wikipedia passages for V3 (CPU, needs internet)

In [ ]:
if "retrieve" in STAGES:
    run("retrieve")

### SLM as verifier: retrieval-grounded (V3) and context-free (V4)

In [ ]:
if "verify" in STAGES:
    run(f"verify --model {SLM} --backend {BACKEND} --dtype {DTYPE}")

### LLM answers

In [ ]:
if "llm" in STAGES:
    run(f"llm --model {LLM} --backend {BACKEND} --dtype {DTYPE}")

### Query embeddings for the query-only prior (V0)

In [ ]:
if "embed" in STAGES:
    run("embed")

### Grade and merge into cache.jsonl

In [ ]:
if "merge" in STAGES:
    run("merge")

### SLM hidden states for the probe signal

In [ ]:
if "hidden" in STAGES:
    run(f"hidden --model {SLM} --dtype {DTYPE}")

## Package the outputs

In [ ]:
os.makedirs(f"{OUT}_zip", exist_ok=True)
for f in STAGE_FILES:
    if os.path.exists(f"{OUT}/{f}"):
        shutil.copy(f"{OUT}/{f}", f"{OUT}_zip/{f}")
shutil.make_archive(f"{WORK}/{os.path.basename(OUT)}", "zip", f"{OUT}_zip")
print("wrote", f"{WORK}/{os.path.basename(OUT)}.zip")